In [0]:
import os
from dotenv import load_dotenv
from functools import reduce
from pyspark.sql.functions import col, count, countDistinct, sum, avg, min, max, stddev, when, desc, year, month, to_date


# Carregamento de variáveis de ambiente
load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")

# Dicionário de credenciais OAUTH para usar com Spark
adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Dicionário de DataSets que serão utilizados
tables = [
    "ecommerce_itens_pedido",
    "food_lotes_producao",
    "food_estoque_lojas",
    "physical_itens_venda_caixa"
]


In [0]:
# Gera os paths das tabelas
def generate_path(layer, table_name=None) -> str: 
    if layer == "silver":
        if table_name.startswith("ecommerce"):
            container = "squad2"
            folder = "grupo2/silver"
        elif table_name.startswith(("food", "physical")):
            container = "squad3"
            folder = "silver"
        else:
            raise ValueError(f"Tabela não reconhecida: {table_name}")
        path = f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/{folder}/{table_name}"  
    else:
        raise ValueError(f"Camada inválida: {layer}")
    return path

# Leitura de datasets
def read_dataset(tables: list, layer="silver") -> dict:
    dataframes = {}
    for table_name in tables:
        df = (
            spark.read
            .format("delta")
            .options(**adls_options)
            .load(generate_path(layer, table_name=table_name))
        )
        dataframes[f"df_{table_name}_{layer}"] = df
    return dataframes

dataframes = read_dataset(tables)

In [0]:
df_ecommerce_itens_pedido_silver = dataframes["df_ecommerce_itens_pedido_silver"]

# ============================================================
# 1. OVERVIEW — ESTRUTURA DA TABELA
# ============================================================

print("========== 1. OVERVIEW ==========")

print(f"Total de registros: {df_ecommerce_itens_pedido_silver.count():,}")
print(f"Total de colunas: {len(df_ecommerce_itens_pedido_silver.columns)}")

df_ecommerce_itens_pedido_silver.printSchema()
display(df_ecommerce_itens_pedido_silver.limit(10))


# ============================================================
# 2. NULL ANALYSIS — VALORES NULOS
# ============================================================

print("========== 2. NULL ANALYSIS ==========")

null_analysis = df_ecommerce_itens_pedido_silver.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_ecommerce_itens_pedido_silver.columns
])

display(null_analysis)


# ============================================================
# 3. DUPLICATE ANALYSIS — REGISTROS DUPLICADOS
# ============================================================

print("========== 3. DUPLICATE ANALYSIS ==========")

total_rows = df_ecommerce_itens_pedido_silver.count()
distinct_rows = df_ecommerce_itens_pedido_silver.distinct().count()

duplicated_keys = (
    df_ecommerce_itens_pedido_silver
    .groupBy("id_item_pedido")
    .count()
    .filter(col("count") > 1)
)

print(f"Linhas completamente duplicadas: {total_rows - distinct_rows:,}")
print(f"Chaves id_item_pedido duplicadas: {duplicated_keys.count():,}")


# ============================================================
# 4. NUMERICAL ANALYSIS — ESTATÍSTICAS DESCRITIVAS
# ============================================================

print("========== 4. NUMERICAL ANALYSIS ==========")

numerical_columns = [
    "quantidade",
    "preco_unitario",
    "desconto_aplicado",
    "receita_bruta",
    "receita_liquida"
]

display(
    df_ecommerce_itens_pedido_silver
    .select(*numerical_columns)
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)


# ============================================================
# 5. NUMERICAL QUALITY — VALORES NEGATIVOS E ZERADOS
# ============================================================

print("========== 5. NUMERICAL QUALITY ==========")

quality_results = []

for column_name in numerical_columns:
    result = df_ecommerce_itens_pedido_silver.agg(
        sum(when(col(column_name) < 0, 1).otherwise(0)).alias("negativos"),
        sum(when(col(column_name) == 0, 1).otherwise(0)).alias("zerados")
    ).first()

    quality_results.append((
        column_name,
        result["negativos"],
        result["zerados"]
    ))

quality_df = spark.createDataFrame(
    quality_results,
    ["coluna", "valores_negativos", "valores_zerados"]
)

display(quality_df)


# ============================================================
# 6. TEMPORAL ANALYSIS — PERÍODO DOS PEDIDOS
# ============================================================

print("========== 6. TEMPORAL ANALYSIS ==========")

display(
    df_ecommerce_itens_pedido_silver.select(
        min("dt_pedido").alias("data_inicial"),
        max("dt_pedido").alias("data_final"),
        countDistinct(
            year("dt_pedido"), month("dt_pedido")
        ).alias("meses_distintos"),
        countDistinct(to_date("dt_pedido")).alias("dias_com_vendas")
    )
)


# ============================================================
# 7. MONTHLY DEMAND — DEMANDA MENSAL
# ============================================================

print("========== 7. MONTHLY DEMAND ==========")

monthly_demand = (
    df_ecommerce_itens_pedido_silver
    .groupBy("ano", "mes")
    .agg(
        sum("quantidade").alias("total_unidades_vendidas"),
        countDistinct("id_pedido").alias("total_pedidos"),
        countDistinct("sku").alias("skus_vendidos"),
        avg("quantidade").alias("media_quantidade_por_item"),
        sum("receita_liquida").alias("receita_liquida_total")
    )
    .orderBy("ano", "mes")
)

display(monthly_demand)


# ============================================================
# 8. SKU ANALYSIS — DEMANDA POR PRODUTO
# ============================================================

print("========== 8. SKU ANALYSIS ==========")

sku_demand = (
    df_ecommerce_itens_pedido_silver
    .groupBy("sku")
    .agg(
        sum("quantidade").alias("total_unidades_vendidas"),
        count("*").alias("total_registros"),
        countDistinct("id_pedido").alias("total_pedidos"),
        avg("preco_unitario").alias("preco_medio"),
        sum("receita_liquida").alias("receita_liquida_total")
    )
    .orderBy(desc("total_unidades_vendidas"))
)

print("Top 20 SKUs por quantidade vendida:")
display(sku_demand.limit(20))

print("Resumo da quantidade de SKUs:")

display(
    sku_demand.agg(
        count("*").alias("total_skus"),
        avg("total_unidades_vendidas").alias("media_unidades_por_sku"),
        max("total_unidades_vendidas").alias("maior_demanda_por_sku"),
        min("total_unidades_vendidas").alias("menor_demanda_por_sku")
    )
)


# ============================================================
# 9. DEMAND DISTRIBUTION — DISTRIBUIÇÃO DAS QUANTIDADES
# ============================================================

print("========== 9. DEMAND DISTRIBUTION ==========")

quantity_distribution = (
    df_ecommerce_itens_pedido_silver
    .groupBy("quantidade")
    .agg(
        count("*").alias("total_registros"),
        sum("quantidade").alias("total_unidades")
    )
    .orderBy("quantidade")
)

display(quantity_distribution)


# ============================================================
# 10. FINAL SUMMARY — RESUMO GERAL
# ============================================================

print("========== 10. FINAL SUMMARY ==========")

display(
    df_ecommerce_itens_pedido_silver.agg(
        count("*").alias("total_registros"),
        countDistinct("id_item_pedido").alias("itens_pedido_distintos"),
        countDistinct("id_pedido").alias("pedidos_distintos"),
        countDistinct("sku").alias("skus_distintos"),
        sum("quantidade").alias("total_unidades_vendidas"),
        sum("receita_bruta").alias("receita_bruta_total"),
        sum("receita_liquida").alias("receita_liquida_total")
    )
)


In [0]:
df_physical_itens_venda_caixa_silver = dataframes["df_physical_itens_venda_caixa_silver"]


from pyspark.sql.functions import col, count, countDistinct, sum, avg, min, max, stddev, when, desc

# ============================================================
# 1. OVERVIEW — ESTRUTURA DA TABELA
# ============================================================

print("========== 1. OVERVIEW ==========")

print(f"Total de registros: {df_physical_itens_venda_caixa_silver.count():,}")
print(f"Total de colunas: {len(df_physical_itens_venda_caixa_silver.columns)}")

df_physical_itens_venda_caixa_silver.printSchema()

display(df_physical_itens_venda_caixa_silver.limit(10))


# ============================================================
# 2. NULL ANALYSIS — VALORES NULOS
# ============================================================

print("========== 2. NULL ANALYSIS ==========")

display(
    df_physical_itens_venda_caixa_silver.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_physical_itens_venda_caixa_silver.columns
    ])
)


# ============================================================
# 3. DUPLICATE ANALYSIS — REGISTROS DUPLICADOS
# ============================================================

print("========== 3. DUPLICATE ANALYSIS ==========")

total_rows = df_physical_itens_venda_caixa_silver.count()
distinct_rows = df_physical_itens_venda_caixa_silver.distinct().count()

duplicated_keys = (
    df_physical_itens_venda_caixa_silver
    .groupBy("id_item_venda")
    .count()
    .filter(col("count") > 1)
)

print(f"Linhas completamente duplicadas: {total_rows - distinct_rows:,}")
print(f"Chaves id_item_venda duplicadas: {duplicated_keys.count():,}")


# ============================================================
# 4. NUMERICAL ANALYSIS — ESTATÍSTICAS DESCRITIVAS
# ============================================================

print("========== 4. NUMERICAL ANALYSIS ==========")

numerical_columns = [
    c for c in [
        "quantidade",
        "preco_unitario_registro",
        "valor_total_item"
    ]
    if c in df_physical_itens_venda_caixa_silver.columns
]

display(
    df_physical_itens_venda_caixa_silver
    .select(*numerical_columns)
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)


# ============================================================
# 5. NUMERICAL QUALITY — VALORES NEGATIVOS E ZERADOS
# ============================================================

print("========== 5. NUMERICAL QUALITY ==========")

quality_results = []

for column_name in numerical_columns:
    result = df_physical_itens_venda_caixa_silver.agg(
        sum(when(col(column_name) < 0, 1).otherwise(0)).alias("negativos"),
        sum(when(col(column_name) == 0, 1).otherwise(0)).alias("zerados")
    ).first()

    quality_results.append((
        column_name,
        result["negativos"],
        result["zerados"]
    ))

quality_df = spark.createDataFrame(
    quality_results,
    ["coluna", "valores_negativos", "valores_zerados"]
)

display(quality_df)


# ============================================================
# 6. PRODUCT ANALYSIS — PRODUTOS MAIS VENDIDOS
# ============================================================

print("========== 6. PRODUCT ANALYSIS ==========")

product_demand = (
    df_physical_itens_venda_caixa_silver
    .groupBy("codigo_barras_produto", "unidade_medida")
    .agg(
        sum("quantidade").alias("total_quantidade_vendida"),
        count("*").alias("total_registros"),
        countDistinct("id_transacao").alias("total_transacoes"),
        sum("valor_total_item").alias("receita_total")
    )
    .orderBy(desc("total_quantidade_vendida"))
)

display(product_demand.limit(20))


# ============================================================
# 7. UNIT ANALYSIS — DISTRIBUIÇÃO POR UNIDADE DE MEDIDA
# ============================================================

print("========== 7. UNIT ANALYSIS ==========")

unit_distribution = (
    df_physical_itens_venda_caixa_silver
    .groupBy("unidade_medida")
    .agg(
        count("*").alias("total_registros"),
        countDistinct("codigo_barras_produto").alias("produtos_distintos"),
        sum("quantidade").alias("quantidade_total"),
        sum("valor_total_item").alias("receita_total")
    )
    .orderBy(desc("total_registros"))
)

display(unit_distribution)


# ============================================================
# 8. TRANSACTION ANALYSIS — ITENS POR TRANSAÇÃO
# ============================================================

print("========== 8. TRANSACTION ANALYSIS ==========")

transaction_analysis = (
    df_physical_itens_venda_caixa_silver
    .groupBy("id_transacao")
    .agg(
        count("*").alias("total_itens"),
        sum("quantidade").alias("quantidade_total"),
        sum("valor_total_item").alias("valor_total_transacao")
    )
)

display(
    transaction_analysis.summary(
        "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
    )
)


# ============================================================
# 9. DEMAND DISTRIBUTION — DISTRIBUIÇÃO DAS QUANTIDADES
# ============================================================

print("========== 9. DEMAND DISTRIBUTION ==========")

quantity_distribution = (
    df_physical_itens_venda_caixa_silver
    .groupBy("quantidade", "unidade_medida")
    .agg(
        count("*").alias("total_registros"),
        sum("valor_total_item").alias("receita_total")
    )
    .orderBy("unidade_medida", "quantidade")
)

display(quantity_distribution)


# ============================================================
# 10. FINAL SUMMARY — RESUMO GERAL
# ============================================================

print("========== 10. FINAL SUMMARY ==========")

display(
    df_physical_itens_venda_caixa_silver.agg(
        count("*").alias("total_registros"),
        countDistinct("id_item_venda").alias("itens_distintos"),
        countDistinct("id_transacao").alias("transacoes_distintas"),
        countDistinct("codigo_barras_produto").alias("produtos_distintos"),
        sum("quantidade").alias("quantidade_total"),
        sum("valor_total_item").alias("receita_total")
    )
)


In [0]:
df_food_estoque_lojas_silver = dataframes["df_food_estoque_lojas_silver"]


from pyspark.sql.functions import col, count, countDistinct, sum, avg, min, max, when, desc, round

# ============================================================
# 1. OVERVIEW — ESTRUTURA DA TABELA
# ============================================================

print("========== 1. OVERVIEW ==========")

df_food_estoque_lojas_silver = df_food_estoque_lojas_silver

total_rows = df_food_estoque_lojas_silver.count()

print(f"Total de registros: {total_rows:,}")
print(f"Total de colunas: {len(df_food_estoque_lojas_silver.columns)}")

df_food_estoque_lojas_silver.printSchema()

display(df_food_estoque_lojas_silver.limit(10))


# ============================================================
# 2. NULL ANALYSIS — VALORES NULOS
# ============================================================

print("========== 2. NULL ANALYSIS ==========")

display(
    df_food_estoque_lojas_silver.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_food_estoque_lojas_silver.columns
    ])
)


# ============================================================
# 3. DUPLICATE ANALYSIS — REGISTROS DUPLICADOS
# ============================================================

print("========== 3. DUPLICATE ANALYSIS ==========")

distinct_rows = df_food_estoque_lojas_silver.distinct().count()

print(f"Linhas completamente duplicadas: {total_rows - distinct_rows:,}")

# Verifica duplicidade considerando loja, SKU, lote e data.
# Essa é uma chave candidata para identificar um registro de estoque.

key_columns = ["id_loja", "sku", "id_lote", "dt_snapshot"]

duplicated_keys = (
    df_food_estoque_lojas_silver
    .groupBy(*key_columns)
    .count()
    .filter(col("count") > 1)
)

print(f"Chaves loja + SKU + lote + snapshot duplicadas: {duplicated_keys.count():,}")

display(duplicated_keys.limit(20))


# ============================================================
# 4. NUMERICAL ANALYSIS — ESTATÍSTICAS DESCRITIVAS
# ============================================================

print("========== 4. NUMERICAL ANALYSIS ==========")

numerical_columns = [
    "quantidade_disponivel",
    "estoque_minimo"
]

display(
    df_food_estoque_lojas_silver
    .select(*numerical_columns)
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)


# ============================================================
# 5. NUMERICAL QUALITY — VALORES NEGATIVOS E ZERADOS
# ============================================================

print("========== 5. NUMERICAL QUALITY ==========")

quality_results = []

for column_name in numerical_columns:
    result = df_food_estoque_lojas_silver.agg(
        sum(when(col(column_name) < 0, 1).otherwise(0)).alias("negativos"),
        sum(when(col(column_name) == 0, 1).otherwise(0)).alias("zerados")
    ).first()

    quality_results.append((
        column_name,
        result["negativos"],
        result["zerados"]
    ))

quality_df = spark.createDataFrame(
    quality_results,
    ["coluna", "valores_negativos", "valores_zerados"]
)

display(quality_df)


# ============================================================
# 6. STOCK ANALYSIS — SITUAÇÃO DO ESTOQUE
# ============================================================

print("========== 6. STOCK ANALYSIS ==========")

df_stock_status = (
    df_food_estoque_lojas_silver
    .withColumn(
        "status_estoque",
        when(col("quantidade_disponivel") == 0, "Sem estoque")
        .when(col("quantidade_disponivel") < col("estoque_minimo"), "Abaixo do mínimo")
        .when(col("quantidade_disponivel") == col("estoque_minimo"), "No mínimo")
        .otherwise("Acima do mínimo")
    )
)

display(
    df_stock_status
    .groupBy("status_estoque")
    .agg(
        count("*").alias("total_registros"),
        countDistinct("sku").alias("produtos_distintos"),
        countDistinct("id_loja").alias("lojas_distintas"),
        sum("quantidade_disponivel").alias("quantidade_total")
    )
    .orderBy(desc("total_registros"))
)


# ============================================================
# 7. STOCKOUT RISK — RISCO DE RUPTURA
# ============================================================

print("========== 7. STOCKOUT RISK ==========")

df_stock_risk = (
    df_stock_status
    .withColumn(
        "risco_ruptura",
        when(col("quantidade_disponivel") <= col("estoque_minimo"), True)
        .otherwise(False)
    )
    .withColumn(
        "quantidade_necessaria",
        when(
            col("quantidade_disponivel") < col("estoque_minimo"),
            col("estoque_minimo") - col("quantidade_disponivel")
        ).otherwise(0)
    )
)

display(
    df_stock_risk
    .groupBy("risco_ruptura")
    .agg(
        count("*").alias("total_registros"),
        countDistinct("sku").alias("produtos_distintos"),
        countDistinct("id_loja").alias("lojas_distintas"),
        sum("quantidade_necessaria").alias("quantidade_para_atingir_minimo")
    )
)


# ============================================================
# 8. PRODUCT ANALYSIS — ESTOQUE POR PRODUTO
# ============================================================

print("========== 8. PRODUCT ANALYSIS ==========")

product_stock = (
    df_food_estoque_lojas_silver
    .groupBy("sku")
    .agg(
        count("*").alias("total_registros"),
        countDistinct("id_loja").alias("total_lojas"),
        sum("quantidade_disponivel").alias("estoque_total"),
        avg("quantidade_disponivel").alias("estoque_medio"),
        avg("estoque_minimo").alias("estoque_minimo_medio")
    )
    .orderBy(desc("estoque_total"))
)

display(product_stock.limit(20))


# ============================================================
# 9. STORE ANALYSIS — ESTOQUE POR LOJA
# ============================================================

print("========== 9. STORE ANALYSIS ==========")

store_stock = (
    df_stock_risk
    .groupBy("id_loja")
    .agg(
        count("*").alias("total_registros"),
        countDistinct("sku").alias("produtos_distintos"),
        sum("quantidade_disponivel").alias("estoque_total"),
        sum(when(col("risco_ruptura"), 1).otherwise(0)).alias("registros_em_risco"),
        sum("quantidade_necessaria").alias("quantidade_para_reposicao")
    )
    .orderBy(desc("registros_em_risco"))
)

display(store_stock)


# ============================================================
# 10. TEMPORAL ANALYSIS — HISTÓRICO DOS SNAPSHOTS
# ============================================================

print("========== 10. TEMPORAL ANALYSIS ==========")

display(
    df_food_estoque_lojas_silver.agg(
        min("dt_snapshot").alias("primeiro_snapshot"),
        max("dt_snapshot").alias("ultimo_snapshot"),
        countDistinct("dt_snapshot").alias("total_datas_snapshot")
    )
)

snapshot_analysis = (
    df_food_estoque_lojas_silver
    .groupBy("dt_snapshot")
    .agg(
        count("*").alias("total_registros"),
        countDistinct("sku").alias("produtos_distintos"),
        countDistinct("id_loja").alias("lojas_distintas"),
        sum("quantidade_disponivel").alias("estoque_total"),
        sum(
            when(
                col("quantidade_disponivel") <= col("estoque_minimo"),
                1
            ).otherwise(0)
        ).alias("registros_no_minimo_ou_abaixo")
    )
    .orderBy("dt_snapshot")
)

display(snapshot_analysis)


# ============================================================
# 11. LOT ANALYSIS — ESTOQUE POR LOTE
# ============================================================

print("========== 11. LOT ANALYSIS ==========")

lot_analysis = (
    df_food_estoque_lojas_silver
    .groupBy("id_lote")
    .agg(
        count("*").alias("total_registros"),
        countDistinct("sku").alias("produtos_distintos"),
        countDistinct("id_loja").alias("lojas_distintas"),
        sum("quantidade_disponivel").alias("estoque_total")
    )
    .orderBy(desc("estoque_total"))
)

display(lot_analysis.limit(20))


# ============================================================
# 12. FINAL SUMMARY — RESUMO GERAL
# ============================================================

print("========== 12. FINAL SUMMARY ==========")

display(
    df_food_estoque_lojas_silver.agg(
        count("*").alias("total_registros"),
        countDistinct("id_loja").alias("lojas_distintas"),
        countDistinct("sku").alias("produtos_distintos"),
        countDistinct("id_lote").alias("lotes_distintos"),
        countDistinct("dt_snapshot").alias("datas_snapshot"),
        sum("quantidade_disponivel").alias("estoque_total"),
        avg("quantidade_disponivel").alias("estoque_medio"),
        sum("estoque_minimo").alias("estoque_minimo_total")
    )
)



In [0]:
df_food_lotes_producao_silver = dataframes["df_food_lotes_producao_silver"]


from pyspark.sql import functions as F

# ============================================================
# 1. OVERVIEW — ESTRUTURA DA TABELA
# ============================================================

print("========== 1. OVERVIEW ==========")

df_lotes = df_food_lotes_producao_silver
colunas = df_lotes.columns

total_registros = df_lotes.count()

print(f"Total de registros: {total_registros:,}")
print(f"Total de colunas: {len(colunas)}")

df_lotes.printSchema()
display(df_lotes.limit(10))


# ============================================================
# 2. NULL ANALYSIS — VALORES NULOS
# ============================================================

print("========== 2. NULL ANALYSIS ==========")

display(
    df_lotes.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in colunas
    ])
)


# ============================================================
# 3. DUPLICATE ANALYSIS — REGISTROS DUPLICADOS
# ============================================================

print("========== 3. DUPLICATE ANALYSIS ==========")

distinct_rows = df_lotes.distinct().count()

print(f"Linhas completamente duplicadas: {total_registros - distinct_rows:,}")

if "id_lote" in colunas:
    duplicated_keys = (
        df_lotes
        .groupBy("id_lote")
        .count()
        .filter(F.col("count") > 1)
    )

    print(f"IDs de lote duplicados: {duplicated_keys.count():,}")
    display(duplicated_keys.limit(20))


# ============================================================
# 4. NUMERICAL ANALYSIS — ESTATÍSTICAS DESCRITIVAS
# ============================================================

print("========== 4. NUMERICAL ANALYSIS ==========")

numerical_columns = [
    c for c in ["quantidade_produzida", "ano", "mes"]
    if c in colunas
]

if numerical_columns:
    display(
        df_lotes.select(*numerical_columns)
        .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
    )


# ============================================================
# 5. NUMERICAL QUALITY — VALORES NEGATIVOS E ZERADOS
# ============================================================

print("========== 5. NUMERICAL QUALITY ==========")

if "quantidade_produzida" in colunas:
    display(
        df_lotes.agg(
            F.sum(
                F.when(F.col("quantidade_produzida") < 0, 1).otherwise(0)
            ).alias("quantidades_negativas"),

            F.sum(
                F.when(F.col("quantidade_produzida") == 0, 1).otherwise(0)
            ).alias("quantidades_zeradas"),

            F.sum(
                F.when(F.col("quantidade_produzida").isNull(), 1).otherwise(0)
            ).alias("quantidades_nulas")
        )
    )


# ============================================================
# 6. STATUS ANALYSIS — SITUAÇÃO DOS LOTES
# ============================================================

print("========== 6. STATUS ANALYSIS ==========")

if "status" in colunas:
    display(
        df_lotes
        .groupBy("status")
        .agg(
            F.count("*").alias("total_lotes"),
            F.sum("quantidade_produzida").alias("quantidade_total"),
            F.avg("quantidade_produzida").alias("quantidade_media")
        )
        .orderBy(F.desc("total_lotes"))
    )


# ============================================================
# 7. TEMPERATURE ANALYSIS — TEMPERATURA DE ARMAZENAMENTO
# ============================================================

print("========== 7. TEMPERATURE ANALYSIS ==========")

if "temperatura_armazenamento_ideal" in colunas:
    display(
        df_lotes
        .groupBy("temperatura_armazenamento_ideal")
        .agg(
            F.count("*").alias("total_lotes"),
            F.sum("quantidade_produzida").alias("quantidade_produzida_total"),
            F.avg("quantidade_produzida").alias("quantidade_media_por_lote")
        )
        .orderBy(F.desc("total_lotes"))
    )


# ============================================================
# 8. SHELF LIFE ANALYSIS — PRAZO DE VALIDADE
# ============================================================

print("========== 8. SHELF LIFE ANALYSIS ==========")

if "dt_fabricacao" in colunas and "dt_validade" in colunas:

    df_lotes_validade = (
        df_lotes
        .withColumn(
            "dias_validade",
            F.datediff("dt_validade", "dt_fabricacao")
        )
        .withColumn(
            "validade_invertida",
            F.col("dt_validade") <= F.col("dt_fabricacao")
        )
    )

    display(
        df_lotes_validade.agg(
            F.min("dias_validade").alias("menor_validade_dias"),
            F.max("dias_validade").alias("maior_validade_dias"),
            F.round(F.avg("dias_validade"), 2).alias("media_validade_dias"),
            F.expr("percentile_approx(dias_validade, 0.5)").alias("mediana_validade_dias"),
            F.sum(
                F.when(F.col("validade_invertida"), 1).otherwise(0)
            ).alias("lotes_com_validade_invertida")
        )
    )

    if "temperatura_armazenamento_ideal" in colunas:
        display(
            df_lotes_validade
            .filter(F.col("dias_validade") > 0)
            .groupBy("temperatura_armazenamento_ideal")
            .agg(
                F.count("*").alias("total_lotes"),
                F.round(F.avg("dias_validade"), 2).alias("validade_media_dias"),
                F.expr(
                    "percentile_approx(dias_validade, 0.5)"
                ).alias("validade_mediana_dias"),
                F.min("dias_validade").alias("validade_minima_dias"),
                F.max("dias_validade").alias("validade_maxima_dias")
            )
        )


# ============================================================
# 9. SUPPLIER ANALYSIS — PRODUÇÃO POR FORNECEDOR
# ============================================================

print("========== 9. SUPPLIER ANALYSIS ==========")

if "id_fornecedor" in colunas:
    supplier_analysis = (
        df_lotes
        .groupBy("id_fornecedor")
        .agg(
            F.count("*").alias("total_lotes"),
            F.sum("quantidade_produzida").alias("producao_total"),
            F.avg("quantidade_produzida").alias("producao_media_por_lote"),
            F.countDistinct("sku").alias("produtos_distintos")
        )
        .orderBy(F.desc("producao_total"))
    )

    display(supplier_analysis)


# ============================================================
# 10. PRODUCT ANALYSIS — PRODUÇÃO POR SKU
# ============================================================

print("========== 10. PRODUCT ANALYSIS ==========")

if "sku" in colunas:
    product_analysis = (
        df_lotes
        .groupBy("sku")
        .agg(
            F.count("*").alias("total_lotes"),
            F.sum("quantidade_produzida").alias("producao_total"),
            F.avg("quantidade_produzida").alias("producao_media"),
            F.countDistinct("id_fornecedor").alias("fornecedores_distintos")
        )
        .orderBy(F.desc("producao_total"))
    )

    display(product_analysis.limit(20))


# ============================================================
# 11. TEMPORAL ANALYSIS — PRODUÇÃO AO LONGO DO TEMPO
# ============================================================

print("========== 11. TEMPORAL ANALYSIS ==========")

if "dt_fabricacao" in colunas:

    display(
        df_lotes.agg(
            F.min("dt_fabricacao").alias("primeira_fabricacao"),
            F.max("dt_fabricacao").alias("ultima_fabricacao"),
            F.countDistinct("dt_fabricacao").alias("datas_distintas")
        )
    )

    monthly_production = (
        df_lotes
        .withColumn("ano_fabricacao", F.year("dt_fabricacao"))
        .withColumn("mes_fabricacao", F.month("dt_fabricacao"))
        .groupBy("ano_fabricacao", "mes_fabricacao")
        .agg(
            F.count("*").alias("total_lotes"),
            F.sum("quantidade_produzida").alias("producao_total"),
            F.avg("quantidade_produzida").alias("producao_media")
        )
        .orderBy("ano_fabricacao", "mes_fabricacao")
    )

    display(monthly_production)


# ============================================================
# 12. DATA QUALITY FLAGS — FLAGS DE QUALIDADE DA SILVER
# ============================================================

print("========== 12. DATA QUALITY FLAGS ==========")

flag_columns = [
    c for c in colunas
    if c.startswith("_flag_")
]

if flag_columns:
    display(
        df_lotes.select([
            F.sum(
                F.when(F.col(c) == True, 1).otherwise(0)
            ).alias(c)
            for c in flag_columns
        ])
    )

    for flag in flag_columns:
        print(f"Distribuição da flag: {flag}")

        display(
            df_lotes
            .groupBy(flag)
            .count()
            .orderBy(F.desc("count"))
        )
else:
    print("Nenhuma coluna de flag encontrada.")


# ============================================================
# 13. FINAL SUMMARY — RESUMO GERAL
# ============================================================

print("========== 13. FINAL SUMMARY ==========")

summary_expressions = [
    F.count("*").alias("total_registros")
]

for c in [
    "id_lote",
    "sku",
    "id_fornecedor",
    "status",
    "temperatura_armazenamento_ideal"
]:
    if c in colunas:
        summary_expressions.append(
            F.countDistinct(c).alias(f"{c}_distintos")
        )

if "quantidade_produzida" in colunas:
    summary_expressions.extend([
        F.sum("quantidade_produzida").alias("quantidade_produzida_total"),
        F.avg("quantidade_produzida").alias("quantidade_media_por_lote")
    ])

display(df_lotes.agg(*summary_expressions))
